# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema available at:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes fields such as accession, description, coverage percentage, peptide counts, molecular weight (MW), calculated isoelectric points (pI), and normalized abundances across samples.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print a quick summary
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a custom object
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their fields using their @id
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set['@id']} | Name: {record_set.get('name', 'N/A')}")
    print("  Fields (@id):")
    for field in record_set['fields']:
        print(f"    - {field['@id']} | Name: {field.get('name', 'N/A')}")
    print()

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. Record set and field `@id`s are used for precise extraction.

In [ ]:
# List all record sets by @id
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Record set @ids found:", record_set_ids)

# Example: Load the primary protein data record set
# (Choose the main record set for protein-level annotation; you may need to check output from previous cell)
protein_record_set_id = record_set_ids[0]  # Adjust if there's more than one

records = list(dataset.records(record_set=protein_record_set_id))
df = pd.DataFrame(records)

print(f"Fields/columns in {protein_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic processing: filtering, normalization, and grouping. We'll use field `@id`s from the overview.

In [ ]:
# Select a meaningful numeric field for analysis.
# Use previous output or the dataset schema for the appropriate @id.
# Example: Assume '@id': 'protein_abundance' for abundance, 'mw' for molecular weight.
numeric_field_id = None
candidate_numeric_fields = [c for c in df.columns if df[c].dtype in ('int64', 'float64')]
print("Numeric candidate fields (choose @id):", candidate_numeric_fields)

# Try to pick one of the common ones, fallback if not present
for candidate in ['protein_abundance', 'abundance', 'MW', 'mw', 'peptide_count']:
    if candidate in df.columns:
        numeric_field_id = candidate
        break
if numeric_field_id is None:
    numeric_field_id = candidate_numeric_fields[0]

print(f"Using numeric field: {numeric_field_id}")

# Filtering high-abundance proteins (example: threshold = 10)
threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered proteins where {numeric_field_id} > {threshold:.2f} (top 5):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} (top 5):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by category (e.g., by 'description' or 'accession' if present)
group_field = None
for candidate in ['description', 'accession', 'protein_name']:
    if candidate in df.columns:
        group_field = candidate
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
    print(f"\nGrouped mean {numeric_field_id} by '{group_field}' (top 5):")
    print(grouped_df.head())
else:
    print("No grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id], bins=30, kde=True, color='steelblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If two numeric columns are present, show a scatterplot
if len(candidate_numeric_fields) >= 2:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(
        x=df[candidate_numeric_fields[0]],
        y=df[candidate_numeric_fields[1]]
    )
    plt.title(f"{candidate_numeric_fields[0]} vs {candidate_numeric_fields[1]}")
    plt.xlabel(candidate_numeric_fields[0])
    plt.ylabel(candidate_numeric_fields[1])
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to load and explore a FAIR-compliant mass spectrometry dataset on protein abundance and related features from human mast cell extracellular vesicles.

- The Croissant schema made the dataset structure explicit and allowed easy field selection by `@id`.
- We loaded records as DataFrames, conducted basic exploratory analysis, normalized quantitative features, and explored groupings (such as by protein description or accession).
- Visualizations helped illuminate data distributions and relationships.

This approach can be adapted to any Croissant-compliant dataset with properly defined record sets and field `@id`s using the `mlcroissant` Python library.